In [ ]:
import os
import random
import glob
import gc
from concurrent.futures import ProcessPoolExecutor

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score
from tqdm.auto import tqdm
from PIL import Image

# ============================================================
# CONFIGURARE RULARE
# ============================================================
# Variante: "auto", "cuda", "tpu", "cpu"
# - pentru Kaggle cu 2x T4: ACCELERATOR = "cuda"
# - pentru Kaggle TPU: ACCELERATOR = "tpu"
ACCELERATOR = "cuda"
SEED = 42

BASE_DIR = '/kaggle/input/datasets/mohlamin/resized-eyepacs-diabetic-retinopathy-dataset'
CSV_PATH_ALL = os.path.join(BASE_DIR, 'all_labels.csv')
OUT_DIR = '/kaggle/working/processed_bg_224'

# Pentru viteza/stabilitate: "efficientnet_b0".
# Alternative utile: "efficientnet_b2", "densenet121", "mobilenet_v3_large", "resnet50".
MODEL_NAME = "efficientnet_b0"


# Marimea seturilor controlate de antrenare.
# Validarea ramane pe distributia reala, ca metricele sa fie mai oneste.
STAGE1_SAMPLES_PER_BINARY_CLASS = 5000
STAGE2_SAMPLES_PER_CLASS = 2500

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def select_device(accelerator="auto"):
    accelerator = accelerator.lower()
    if accelerator == "tpu":
        try:
            import torch_xla
            import torch_xla.core.xla_model as xm
        except ModuleNotFoundError as exc:
            raise RuntimeError(
                "Ai ales ACCELERATOR='tpu', dar mediul curent nu are torch_xla. "
                "Pe Kaggle cu 2x T4 seteaza ACCELERATOR='cuda'. "
                "Foloseste 'tpu' doar daca ai pornit un notebook Kaggle cu accelerator TPU."
            ) from exc
        return torch_xla.device(), True, xm

    if accelerator == "cuda" or (accelerator == "auto" and torch.cuda.is_available()):
        if not torch.cuda.is_available():
            raise RuntimeError("Ai ales CUDA, dar torch.cuda.is_available() este False.")
        return torch.device("cuda"), False, None

    return torch.device("cpu"), False, None

seed_everything(SEED)
device, IS_TPU, xm = select_device(ACCELERATOR)

print(f"Device curent: {device}")
print(f"TPU activ: {IS_TPU}")
if device.type == "cuda":
    print(f"GPU-uri disponibile: {torch.cuda.device_count()} | {torch.cuda.get_device_name(0)}")

In [ ]:
print("Se mapeaza datasetul complet...")

df = pd.read_csv(CSV_PATH_ALL)
print(f"Linii in all_labels.csv: {len(df)}")
print("Coloane CSV:", list(df.columns))

label_candidates = ['level', 'diagnosis', 'label', 'class', 'dr_level', 'retinopathy_grade']
image_candidates = ['image', 'image_id', 'id_code', 'filename', 'file_name', 'name']

label_col = next((col for col in label_candidates if col in df.columns), None)
image_col = next((col for col in image_candidates if col in df.columns), None)

if label_col is None:
    raise KeyError(
        "Nu gasesc coloana de eticheta. Coloane disponibile: "
        f"{list(df.columns)}. Redenumeste coloana de diagnostic in 'level' sau adaug-o in label_candidates."
    )
if image_col is None:
    raise KeyError(
        "Nu gasesc coloana cu numele imaginii. Coloane disponibile: "
        f"{list(df.columns)}. Redenumeste coloana imaginii in 'image' sau adaug-o in image_candidates."
    )

df = df.rename(columns={label_col: 'level', image_col: 'image'})
df['level'] = pd.to_numeric(df['level'], errors='coerce')
df = df.dropna(subset=['level']).copy()
df['level'] = df['level'].astype(int)
df['image_key'] = df['image'].astype(str).apply(lambda x: os.path.splitext(os.path.basename(x))[0])

toate_imaginile = glob.glob(os.path.join(BASE_DIR, '**', '*.*'), recursive=True)
toate_imaginile = [p for p in toate_imaginile if p.lower().endswith(('.png', '.jpg', '.jpeg'))]
mapare_cai = {os.path.splitext(os.path.basename(p))[0]: p for p in toate_imaginile}

df['file_path'] = df['image_key'].map(mapare_cai)
df = df.dropna(subset=['file_path']).reset_index(drop=True)
df['original_file_path'] = df['file_path']

print("Curatam intrarile catre fisiere lipsa/corupte...")
indici_valizi = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scanare integritate"):
    try:
        if os.path.getsize(row['file_path']) > 0:
            indici_valizi.append(idx)
    except OSError:
        pass

df = df.loc[indici_valizi].reset_index(drop=True)
df['is_diseased'] = df['level'].apply(lambda x: 0 if x == 0 else 1)
df['level_stage2'] = df['level'].apply(lambda x: x - 1 if x > 0 else -1)

print(f"Imagini valide fizic: {len(df)}")
print(df['level'].value_counts().sort_index())

# Preprocesarea Ben Graham (Multithreaded pe TOATE imaginile)

In [ ]:
print("=== PREPROCESARE OFFLINE BEN GRAHAM + CLAHE ===")

cv2.setNumThreads(0)
os.makedirs(OUT_DIR, exist_ok=True)

def process_and_save(args):
    img_path, image_id = args
    out_path = os.path.join(OUT_DIR, f"{image_id}.jpg")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return out_path

    try:
        img = cv2.imread(img_path)
        if img is None:
            return None

        size = 224
        img_resized = cv2.resize(img, (size, size))
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

        _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        kernel_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        thresh_clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)
        contours, _ = cv2.findContours(thresh_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            cv2.imwrite(out_path, img_resized)
            return out_path

        contur_ochi = max(contours, key=cv2.contourArea)
        masca_ochi = np.zeros((size, size), dtype=np.uint8)
        cv2.drawContours(masca_ochi, [contur_ochi], -1, 255, thickness=cv2.FILLED)

        kernel_eroziune = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
        masca_ochi = cv2.erode(masca_ochi, kernel_eroziune, iterations=1)

        sigmaX = size / 30.0
        blur = cv2.GaussianBlur(gray, (0, 0), sigmaX)
        ben_graham_gray = cv2.addWeighted(gray, 4, blur, -4, 128)

        masca_exterior = cv2.bitwise_not(masca_ochi)
        dist = cv2.distanceTransform(masca_exterior, cv2.DIST_L2, 5)
        halo_width = 35.0
        rezultat_final = np.full((size, size), 128.0, dtype=np.float32)

        halo_mask = (dist > 0) & (dist <= halo_width)
        rezultat_final[halo_mask] = 128.0 * (dist[halo_mask] / halo_width)
        rezultat_final[masca_ochi == 255] = ben_graham_gray[masca_ochi == 255]

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        rezultat_final = clahe.apply(rezultat_final.astype(np.uint8))
        rezultat_final_3c = cv2.cvtColor(rezultat_final, cv2.COLOR_GRAY2BGR)
        cv2.imwrite(out_path, rezultat_final_3c)
        return out_path
    except Exception:
        return None

tasks = [(row['original_file_path'], row['image_key']) for _, row in df.iterrows()]
rezultate_finale = []

BATCH_SIZE_MP = 5000
WORKERS = min(4, os.cpu_count() or 1)

print(f"Procesam {len(tasks)} imagini in {OUT_DIR}...")
for i in range(0, len(tasks), BATCH_SIZE_MP):
    batch_tasks = tasks[i : i + BATCH_SIZE_MP]
    with ProcessPoolExecutor(max_workers=WORKERS) as executor:
        batch_rezultate = list(tqdm(
            executor.map(process_and_save, batch_tasks, chunksize=50),
            total=len(batch_tasks),
            desc=f"Batch {i // BATCH_SIZE_MP + 1}"
        ))
    rezultate_finale.extend(batch_rezultate)
    del batch_tasks, batch_rezultate
    gc.collect()

df['processed_path'] = rezultate_finale
df = df.dropna(subset=['processed_path']).reset_index(drop=True)
df['file_path'] = df['processed_path']
df = df.drop(columns=['processed_path'])

# Important: splitul se face DUPA ce file_path indica imaginile procesate.
df_train_full, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['level']
)
df_train_full = df_train_full.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

def sample_exact(dataframe, n, random_state, label):
    replace = len(dataframe) < n
    if replace:
        print(f"Atentie: {label} are doar {len(dataframe)} imagini; folosesc sample cu replacement pana la {n}.")
    return dataframe.sample(n=n, replace=replace, random_state=random_state)

# Stage 1: train controlat 5000 sanatosi + 5000 bolnavi.
df_train_s1_no_dr = sample_exact(
    df_train_full[df_train_full['is_diseased'] == 0],
    STAGE1_SAMPLES_PER_BINARY_CLASS,
    SEED,
    "Stage 1 No DR"
)
df_train_s1_dr = sample_exact(
    df_train_full[df_train_full['is_diseased'] == 1],
    STAGE1_SAMPLES_PER_BINARY_CLASS,
    SEED + 1,
    "Stage 1 DR"
)
df_train_s1 = pd.concat([df_train_s1_no_dr, df_train_s1_dr], ignore_index=True)
df_train_s1 = df_train_s1.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Stage 2: train controlat 2500 pentru fiecare clasa 1-4.
stage2_parts = []
for level_stage2 in range(4):
    df_level = df_train_full[df_train_full['level_stage2'] == level_stage2]
    stage2_parts.append(sample_exact(
        df_level,
        STAGE2_SAMPLES_PER_CLASS,
        SEED + 10 + level_stage2,
        f"Stage 2 clasa {level_stage2}"
    ))
df_train_s2 = pd.concat(stage2_parts, ignore_index=True)
df_train_s2 = df_train_s2.sample(frac=1, random_state=SEED).reset_index(drop=True)

df_train_stage2 = df_train_full[df_train_full['is_diseased'] == 1].reset_index(drop=True)
df_val_stage2 = df_val[df_val['is_diseased'] == 1].reset_index(drop=True)

print(f"Gata: {len(df)} imagini procesate.")
print("\n=== SPLITURI NATURALE DUPA PREPROCESARE ===")
print(f"Train full: {len(df_train_full)} | Val full: {len(df_val)}")
print("Train full - distributie 5 clase:")
print(df_train_full['level'].value_counts().sort_index())
print("Val full - distributie 5 clase:")
print(df_val['level'].value_counts().sort_index())
print("Train full - distributie binara Stage 1:")
print(df_train_full['is_diseased'].value_counts().sort_index())
print("Val full - distributie binara Stage 1:")
print(df_val['is_diseased'].value_counts().sort_index())

print("\n=== SETURI CONTROLATE DE ANTRENARE ===")
print(f"Stage 1 train controlat: {len(df_train_s1)} | Val Stage 1: {len(df_val)}")
print(df_train_s1['is_diseased'].value_counts().sort_index())
print(f"Stage 2 train controlat: {len(df_train_s2)} | Val Stage 2: {len(df_val_stage2)}")
print(df_train_s2['level_stage2'].value_counts().sort_index())
print("Val Stage 2 - distributie:")
print(df_val_stage2['level_stage2'].value_counts().sort_index())

In [ ]:
print("=== DATASETS SI DATALOADERS ===")

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class RetinopathyDataset(Dataset):
    def __init__(self, dataframe, transform=None, is_stage2=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.is_stage2 = is_stage2

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['file_path']).convert("RGB")
        if self.transform:
            image = self.transform(image)

        if self.is_stage2:
            return image, torch.tensor(row['level_stage2'], dtype=torch.long)
        return image, torch.tensor(row['is_diseased'], dtype=torch.float32)

BATCH_SIZE = 64
NUM_WORKERS = 0 if IS_TPU else 4
PIN_MEMORY = (device.type == "cuda")

loader_train_s1 = DataLoader(
    RetinopathyDataset(df_train_s1, transform_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
loader_val_s1 = DataLoader(
    RetinopathyDataset(df_val, transform_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
loader_train_s2 = DataLoader(
    RetinopathyDataset(df_train_s2, transform_train, is_stage2=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
loader_val_s2 = DataLoader(
    RetinopathyDataset(df_val_stage2, transform_val, is_stage2=True),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f"Batch-uri Stage 1: train={len(loader_train_s1)} | val={len(loader_val_s1)}")
print(f"Batch-uri Stage 2: train={len(loader_train_s2)} | val={len(loader_val_s2)}")
print("\nNumar imagini folosite de DataLoader:")
print(f"Stage 1 train: {len(loader_train_s1.dataset)} | Stage 1 val: {len(loader_val_s1.dataset)}")
print(f"Stage 2 train: {len(loader_train_s2.dataset)} | Stage 2 val: {len(loader_val_s2.dataset)}")

In [ ]:
print("=== VERIFICARE VIZUALA: PREPROCESARE SI AUGMENTARE ===")

def show_image_pair(img_a, img_b, title_a, title_b, figsize=(10, 5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].imshow(img_a)
    axes[0].set_title(title_a)
    axes[0].axis("off")
    axes[1].imshow(img_b)
    axes[1].set_title(title_b)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

def tensor_to_image(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    tensor = tensor.detach().cpu() * std + mean
    tensor = tensor.clamp(0, 1)
    return tensor.permute(1, 2, 0).numpy()

sample_row = df_train_s1.sample(1, random_state=SEED).iloc[0]
img_originala = Image.open(sample_row['original_file_path']).convert("RGB")
img_procesata = Image.open(sample_row['file_path']).convert("RGB")

show_image_pair(
    img_originala,
    img_procesata,
    f"Inainte de preprocesare | nivel {sample_row['level']}",
    "Dupa preprocesare Ben Graham + CLAHE"
)

img_procesata_pil = Image.open(sample_row['file_path']).convert("RGB")
img_fara_aug = tensor_to_image(transform_val(img_procesata_pil))
img_cu_aug = tensor_to_image(transform_train(img_procesata_pil))

show_image_pair(
    img_fara_aug,
    img_cu_aug,
    "Inainte de augmentare",
    "Dupa augmentare dinamica"
)

In [ ]:
print("=== MODELE, LOSS, OPTIMIZER, SCHEDULER ===")

EPOCHS = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.30

def replace_classifier(model, num_classes):
    if hasattr(model, "fc"):
        num_ftrs = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(p=DROPOUT), nn.Linear(num_ftrs, num_classes))
    elif hasattr(model, "classifier"):
        classifier = model.classifier
        if isinstance(classifier, nn.Sequential):
            in_features = None
            for layer in reversed(classifier):
                if isinstance(layer, nn.Linear):
                    in_features = layer.in_features
                    break
            if in_features is None:
                raise ValueError("Nu gasesc strat Linear in classifier.")
            model.classifier = nn.Sequential(nn.Dropout(p=DROPOUT), nn.Linear(in_features, num_classes))
        elif isinstance(classifier, nn.Linear):
            model.classifier = nn.Sequential(nn.Dropout(p=DROPOUT), nn.Linear(classifier.in_features, num_classes))
        else:
            raise ValueError(f"Classifier necunoscut pentru {type(model).__name__}.")
    else:
        raise ValueError(f"Nu stiu sa inlocuiesc head-ul pentru {type(model).__name__}.")
    return model

def build_model(num_classes, model_name=MODEL_NAME):
    model_name = model_name.lower()

    if model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    elif model_name == "efficientnet_b2":
        model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    elif model_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    elif model_name == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    else:
        raise ValueError(
            "MODEL_NAME invalid. Alege dintre: resnet50, efficientnet_b0, "
            "efficientnet_b2, densenet121, mobilenet_v3_large."
        )

    model = replace_classifier(model, num_classes)
    model = model.to(device)

    if device.type == "cuda" and torch.cuda.device_count() > 1:
        print(f"Folosim DataParallel pe {torch.cuda.device_count()} GPU-uri.")
        model = nn.DataParallel(model)
    return model

print(f"Arhitectura folosita: {MODEL_NAME}")
model_stage1 = build_model(num_classes=1)
model_stage2 = build_model(num_classes=4)

neg_count = int((df_train_s1['is_diseased'] == 0).sum())
pos_count = int((df_train_s1['is_diseased'] == 1).sum())
pos_weight = torch.tensor([neg_count / max(pos_count, 1)], dtype=torch.float32, device=device)
criterion_s1 = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

stage2_counts = df_train_s2['level_stage2'].value_counts().reindex(range(4), fill_value=1).sort_index()
class_weights = len(df_train_s2) / (4.0 * torch.tensor(stage2_counts.values, dtype=torch.float32))
criterion_s2 = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.05)

optimizer_s1 = optim.AdamW(model_stage1.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
optimizer_s2 = optim.AdamW(model_stage2.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler_s1 = optim.lr_scheduler.CosineAnnealingLR(optimizer_s1, T_max=EPOCHS)
scheduler_s2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_s2, T_max=EPOCHS)

print(f"Stage 1 pos_weight: {pos_weight.item():.4f}")
print("Stage 2 class weights:", class_weights.tolist())

In [ ]:
print("=== MOTOR DE ANTRENARE CUDA/TPU ===")

def _state_dict_cpu(model):
    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    return {k: v.detach().cpu().clone() for k, v in state.items()}

def _load_state_dict(model, state):
    target = model.module if isinstance(model, nn.DataParallel) else model
    target.load_state_dict(state)

def optimizer_step(optimizer):
    if IS_TPU:
        xm.optimizer_step(optimizer, barrier=True)
    else:
        optimizer.step()

def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler=None,
    epochs=10,
    is_binary=True,
    patience=4,
    min_delta=1e-4
):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_qwk': []}
    best_score = -float("inf")
    best_state = None
    bad_epochs = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_preds, train_targets = 0.0, [], []

        for images, labels in tqdm(train_loader, desc=f"Epoca {epoch + 1}/{epochs} [TRAIN]"):
            images = images.to(device, non_blocking=PIN_MEMORY)
            labels = labels.to(device, non_blocking=PIN_MEMORY)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            if is_binary:
                outputs = outputs.squeeze(1)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_step(optimizer)

            train_loss += loss.item() * images.size(0)
            if is_binary:
                preds = (torch.sigmoid(outputs) > 0.5).long()
            else:
                preds = torch.argmax(outputs, dim=1)

            train_preds.extend(preds.detach().cpu().numpy())
            train_targets.extend(labels.detach().cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(train_targets, train_preds)

        model.eval()
        val_loss, val_preds, val_targets = 0.0, [], []

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoca {epoch + 1}/{epochs} [VAL]"):
                images = images.to(device, non_blocking=PIN_MEMORY)
                labels = labels.to(device, non_blocking=PIN_MEMORY)
                outputs = model(images)
                if is_binary:
                    outputs = outputs.squeeze(1)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                if is_binary:
                    preds = (torch.sigmoid(outputs) > 0.5).long()
                else:
                    preds = torch.argmax(outputs, dim=1)

                val_preds.extend(preds.detach().cpu().numpy())
                val_targets.extend(labels.detach().cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_acc = accuracy_score(val_targets, val_preds)
        val_qwk = cohen_kappa_score(val_targets, val_preds, weights='quadratic')

        if scheduler is not None:
            scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        print(
            f"Epoca {epoch + 1:02d} | TL: {train_loss:.4f} | VL: {val_loss:.4f} | "
            f"T_Acc: {train_acc:.4f} | V_Acc: {val_acc:.4f} | Val QWK: {val_qwk:.4f} | LR: {current_lr:.2e}"
        )

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_qwk'].append(val_qwk)

        if val_qwk > best_score + min_delta:
            best_score = val_qwk
            best_state = _state_dict_cpu(model)
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping: QWK nu s-a imbunatatit timp de {patience} epoci.")
                break

    if best_state is not None:
        _load_state_dict(model, best_state)
        print(f"Am restaurat cel mai bun model dupa Val QWK: {best_score:.4f}")

    return history

In [ ]:
print("\n" + "=" * 50)
print("ANTRENARE STAGE 1: detectie boala")
print("=" * 50)
history_s1 = train_model(
    model_stage1,
    loader_train_s1,
    loader_val_s1,
    criterion_s1,
    optimizer_s1,
    scheduler=scheduler_s1,
    epochs=EPOCHS,
    is_binary=True,
    patience=4
)

print("\n" + "=" * 50)
print("ANTRENARE STAGE 2: severitate 1-4")
print("=" * 50)
history_s2 = train_model(
    model_stage2,
    loader_train_s2,
    loader_val_s2,
    criterion_s2,
    optimizer_s2,
    scheduler=scheduler_s2,
    epochs=EPOCHS,
    is_binary=False,
    patience=4
)

In [ ]:
def plot_metrics(history, title):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Loss
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Val Loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].legend()
    
    # 2. Accuracy
    axes[1].plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    axes[1].plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].legend()
    
    # 3. QWK
    axes[2].plot(epochs, history['val_qwk'], 'g-', label='Val QWK', marker='o')
    axes[2].set_title(f'{title} - Quadratic Weighted Kappa')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()

print("\n📊 GRAFICE REZULTATE:")
plot_metrics(history_s1, "STAGE 1 (ResNet50 Binar)")
plot_metrics(history_s2, "STAGE 2 (ResNet50 Multiclasă)")

In [ ]:
from PIL import Image
import random

print("=== 🔬 ETAPA 8: TESTARE INTERACTIVĂ (Predicție vs Adevăr) ===")

def testeaza_imagine(image_path, model_s1, model_s2, transform, device, adevar_text=None):
    """Evaluează o imagine trecând-o prin tot pipeline-ul de diagnostic."""
    
    # 1. Încărcarea imaginii
    try:
        img_originala = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"❌ Eroare la încărcarea imaginii: {e}")
        return

    # Afișăm imaginea originală pe ecran cu eticheta reală
    plt.figure(figsize=(5, 5))
    plt.imshow(img_originala)
    
    titlu = "Imagine introdusă pentru test"
    if adevar_text:
        titlu += f"\n[ DIAGNOSTIC REAL (CSV): {adevar_text} ]"
        
    plt.title(titlu, color='darkblue', fontweight='bold')
    plt.axis('off')
    plt.show()

    # 2. Preprocesare (Advanced Transform + Tensor)
    img_tensor = transform(img_originala).unsqueeze(0).to(device)

    # 3. Predicția
    model_s1.eval()
    model_s2.eval()

    with torch.no_grad():
        print("🤖 Modelul analizează retina...")
        
        # --- STAGE 1 (Detecție Boală) ---
        out_s1 = model_s1(img_tensor).squeeze(1)
        prob_bolnav = torch.sigmoid(out_s1).item()
        
        print(f"   [Stage 1] Probabilitate detectată pentru anomalii: {prob_bolnav*100:.2f}%")

        if prob_bolnav < 0.5:
            print("\n✅ PREDICȚIE MODEL: Ochi Sănătos (No DR)")
        else:
            print("\n⚠️ PREDICȚIE MODEL: S-au detectat anomalii. Se trece la evaluarea severității...")
            
            # --- STAGE 2 (Severitate) ---
            out_s2 = model_s2(img_tensor)
            probabilitati_clase = torch.softmax(out_s2, dim=1)[0]
            clasa_prezisa = torch.argmax(out_s2, dim=1).item()
            
            nume_clase = ['Mild (Stadiul 1)', 'Moderate (Stadiul 2)', 'Severe (Stadiul 3)', 'Proliferative (Stadiul 4)']
            nume_clasa = nume_clase[clasa_prezisa]
            prob_clasa = probabilitati_clase[clasa_prezisa].item() * 100
            
            print(f"🚨 PREDICȚIE FINALĂ MODEL: Bolnav -> {nume_clasa} (Încredere: {prob_clasa:.2f}%)")
            
        print("-" * 50)
        if adevar_text:
            print(f"🩺 ADEVĂRUL MEDICAL (din date): {adevar_text}")

# ==========================================
# --- ZONĂ DE TESTARE ---
# ==========================================

# Alegem o imagine ALEATORIE din setul de validare
rand_idx = random.randint(0, len(df_val) - 1)
rand_row = df_val.iloc[rand_idx]

cale_imagine_test = rand_row['file_path']

# Construim string-ul cu adevărul absolut pe baza etichetelor din CSV
if rand_row['is_diseased'] == 0:
    adevar = "Sănătos (Nivel 0)"
else:
    # Dacă e bolnav, extragem stadiul exact
    adevar = f"Bolnav - Stadiul {rand_row['level']}"

print(f"Testăm fișierul: {cale_imagine_test}")

# Apelăm funcția dându-i și parametrul adevar_text
testeaza_imagine(cale_imagine_test, model_stage1, model_stage2, transform_val, device, adevar_text=adevar)

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import torch
import numpy as np

print("=== 📊 ETAPA 7: EVALUARE ȘI REZULTATE ===")

def plot_history(history, title):
    """Funcție utilitară pentru a desena graficele de antrenare (PyTorch)"""
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Graficul de Loss
    ax1.plot(epochs, history['train_loss'], label='Train Loss')
    ax1.plot(epochs, history['val_loss'], label='Val Loss')
    ax1.set_title(f'{title} - Loss')
    ax1.set_xlabel('Epoci')
    ax1.legend()
    
    # Graficul de Accuracy și QWK
    ax2.plot(epochs, history['train_acc'], label='Train Acc')
    ax2.plot(epochs, history['val_acc'], label='Val Acc')
    if 'val_qwk' in history:
        ax2.plot(epochs, history['val_qwk'], label='Val QWK', linestyle='--')
    ax2.set_title(f'{title} - Performanță')
    ax2.set_xlabel('Epoci')
    ax2.legend()
    
    plt.show()

def evaluate_pytorch_model(model, val_loader, is_binary=False, class_names=None):
    """Funcție utilitară pentru Matricea de Confuzie și Raport (PyTorch)"""
    model.eval()
    y_true = []
    y_pred = []
    
    print("⏳ Se generează predicțiile pentru setul de validare...")
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            
            if is_binary:
                outputs = outputs.squeeze(1)
                preds = (torch.sigmoid(outputs) > 0.5).int()
            else:
                preds = torch.argmax(outputs, dim=1)
                
            y_pred.extend(preds.cpu().numpy())
            y_true.extend(labels.numpy())
            
    # Afișăm raportul
    print("\n📝 Raport de Clasificare:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    # Desenăm Matricea de Confuzie
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Prezis')
    plt.ylabel('Adevărat')
    plt.title('Matrice de Confuzie')
    plt.show()

# 1. Rezultate STAGE 1 (ResNet50 Binar)
print("\n" + "="*40)
print("REZULTATE STAGE 1: DETECȚIE (Sănătos vs Bolnav)")
print("="*40)
plot_history(history_s1, "Stage 1: ResNet50 (Detecție)")
evaluate_pytorch_model(model_stage1, loader_val_s1, is_binary=True, class_names=['Sănătos (0)', 'Bolnav (1-4)'])

# 2. Rezultate STAGE 2 (ResNet50 Multiclasă)
print("\n" + "="*40)
print("REZULTATE STAGE 2: CLASIFICARE SEVERITATE (Stadiile 1-4)")
print("="*40)
plot_history(history_s2, "Stage 2: ResNet50 (Severitate)")
stage2_classes = ['Mild (Stadiul 1)', 'Moderate (Stadiul 2)', 'Severe (Stadiul 3)', 'Proliferative (Stadiul 4)']
evaluate_pytorch_model(model_stage2, loader_val_s2, is_binary=False, class_names=stage2_classes)